In [1]:
# Sanity check: make sure every package we'll need actually imports
import sys
print(f"Python {sys.version.split()[0]}\n")

# Core data tools
import numpy as np
import pandas as pd

# Corpus + PDF parsing
import arxiv
import fitz  # this is what 'pymupdf' is called when you import it

# Embeddings + vector store + sparse retrieval
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from rank_bm25 import BM25Okapi

# Document store (in-memory)
import mongomock

# AutoML
import optuna

# Online learning
import river

print("✓ All imports successful\n")
print(f"  numpy      : {np.__version__}")
print(f"  pandas     : {pd.__version__}")
print(f"  optuna     : {optuna.__version__}")
print(f"  river      : {river.__version__}")
print(f"  mongomock  : {mongomock.__version__}")

Python 3.13.13

✓ All imports successful

  numpy      : 2.4.4
  pandas     : 2.3.3
  optuna     : 4.8.0
  river      : 0.24.2
  mongomock  : 4.3.0


In [2]:
# Configuration — paths and search parameters for our corpus
from pathlib import Path

# Figure out the project root regardless of where the notebook runs from
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR     = PROJECT_ROOT / "data"
PAPERS_DIR   = DATA_DIR / "papers"
METADATA_FILE = DATA_DIR / "corpus_metadata.json"

PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Search settings
SEARCH_QUERY = '"retrieval augmented generation" OR "dense retrieval"'
MAX_PAPERS = 60

print(f"Project root  : {PROJECT_ROOT}")
print(f"Papers dir    : {PAPERS_DIR}")
print(f"Metadata file : {METADATA_FILE}")
print(f"\nSearch query  : {SEARCH_QUERY}")
print(f"Target count  : {MAX_PAPERS} papers")

Project root  : /Users/gamzeokmen/Documents/csai415-paper-rag
Papers dir    : /Users/gamzeokmen/Documents/csai415-paper-rag/data/papers
Metadata file : /Users/gamzeokmen/Documents/csai415-paper-rag/data/corpus_metadata.json

Search query  : "retrieval augmented generation" OR "dense retrieval"
Target count  : 60 papers


In [5]:
import arxiv

# very patient client - small pages, long waits
client = arxiv.Client(
    page_size=10,
    delay_seconds=15,
    num_retries=8,
)

# 40 is enough for D1, less stress on the API
TARGET = 40

search = arxiv.Search(
    query=SEARCH_QUERY,
    max_results=TARGET,
    sort_by=arxiv.SortCriterion.Relevance,
)

# fetch one at a time so partial success isn't lost
results = []
try:
    for p in client.results(search):
        results.append(p)
        print(f"  [{len(results):>2}] {p.get_short_id():<14} {p.title[:60]}")
except Exception as e:
    print(f"\nstopped: {type(e).__name__}: {e}")
    print(f"have {len(results)} papers - if >=20, that's workable")

print(f"\ntotal: {len(results)}")

  [ 1] 2411.18583v1   Automated Literature Review Using NLP Techniques and LLM-Bas
  [ 2] 2602.07739v1   HypRAG: Hyperbolic Dense Retrieval for Retrieval Augmented G
  [ 3] 2502.00306v2   Riddle Me This! Stealthy Membership Inference for Retrieval-
  [ 4] 2510.22344v1   FAIR-RAG: Faithful Adaptive Iterative Refinement for Retriev
  [ 5] 2601.05264v1   Engineering the RAG Stack: A Comprehensive Review of the Arc
  [ 6] 2503.16581v1   Investigating Retrieval-Augmented Generation in Quranic Stud
  [ 7] 2602.00899v1   Domain-Adaptive and Scalable Dense Retrieval for Content-Bas
  [ 8] 2502.01113v3   GFM-RAG: Graph Foundation Model for Retrieval Augmented Gene
  [ 9] 2412.05838v1   A Collaborative Multi-Agent Approach to Retrieval-Augmented 
  [10] 2603.09891v1   Overview of the TREC 2025 Retrieval Augmented Generation (RA

stopped: HTTPError: Page request resulted in HTTP 429 (https://export.arxiv.org/api/query?search_query=%22retrieval+augmented+generation%22+OR+%22dense+retrieval%22&id_l

In [8]:
import json
import ssl
import urllib.request
from datetime import datetime
from tqdm.notebook import tqdm

# bypass SSL verification (temporary workaround for mac cert issues)
ssl_context = ssl._create_unverified_context()

metadata = []

for paper in tqdm(results, desc="downloading"):
    short_id = paper.get_short_id().split('v')[0]
    pdf_name = f"{short_id}.pdf"
    pdf_path = PAPERS_DIR / pdf_name
    
    if pdf_path.exists():
        print(f"  already have {short_id}")
    else:
        try:
            # download using urllib with ssl bypass
            pdf_url = paper.pdf_url
            req = urllib.request.Request(pdf_url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, context=ssl_context) as response:
                pdf_path.write_bytes(response.read())
            print(f"  got {short_id}")
        except Exception as e:
            print(f"  skip {short_id}: {e}")
            continue
    
    metadata.append({
        "doc_id": short_id,
        "title": paper.title,
        "authors": [a.name for a in paper.authors],
        "abstract": paper.summary,
        "year": paper.published.year,
        "primary_category": paper.primary_category,
        "pdf_filename": pdf_name,
        "ingested_at": datetime.now().isoformat(),
    })

with open(METADATA_FILE, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\ndone. {len(metadata)} papers")

downloading:   0%|          | 0/10 [00:00<?, ?it/s]

  got 2411.18583
  got 2602.07739
  got 2502.00306
  got 2510.22344
  got 2601.05264
  got 2503.16581
  got 2602.00899
  got 2502.01113
  got 2412.05838
  got 2603.09891

done. 10 papers
